# Notebook 3: Normalizing Flows from Scratch

In this notebook, we will implement a **Normalizing Flow** from scratch using PyTorch. Normalizing flows are a powerful class of generative models that allow for exact density estimation and sampling.

**What you will learn**
- the change of variables formula, and why the Jacobian matters
- how coupling layers give a transformation which is flexible, invertible and has a cheap Jacobian (RealNVP)
- how to train a flow, sample from it, look inside it and test it
- how Masked Autoregressive Flows (MAF) and Neural Spline Flows (NSF) improve on RealNVP
- how to make a flow conditional, which is the building block of simulation-based inference

**Runtime:** about one minute on a laptop.

## Concept

At the end of Notebook 2, a network predicted a probability distribution, but it was restricted to a Gaussian. A normalizing flow removes this restriction: it transforms a simple base distribution $p_z(z)$ (usually a standard Gaussian) into a complex target distribution $p_x(x)$ using a sequence of invertible transformations $f$.

$$ x = f(z) $$
$$ z = f^{-1}(x) $$

### Change of Variables Formula

The probability density of the transformed variable $x$ is given by:

$$ \log p_x(x) = \log p_z(z) - \log |\det J_f(z)| = \log p_z(f^{-1}(x)) + \log |\det J_{f^{-1}}(x)| $$

Where $J_f(z) = \partial x / \partial z$ is the Jacobian of the transformation $f$ and $J_{f^{-1}}(x) = \partial z / \partial x$ is the Jacobian of its inverse.

To evaluate the density of a data point we therefore need the *normalizing* direction $x \to z$ and its Jacobian determinant, while for sampling we need the *generative* direction $z \to x$.

**Convention in this notebook:** the `forward` method of all our models implements the normalizing direction $x \to z$ and returns $\log |\det \partial z / \partial x|$. The `inverse` method implements the generative direction $z \to x$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.datasets import make_moons
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Our models are tiny, so they do not benefit from many CPU threads. Limiting the thread
# count avoids large slow-downs from thread oversubscription (common on laptops).
torch.set_num_threads(4)

## 1. Warm-up: Change of Variables in 1D

Before we build a flow, let us check the change of variables formula in the simplest possible case: one dimension and a fixed transformation without any neural network.

We take $z \sim \mathcal{N}(0, 1)$ and $x = f(z) = e^z$. The inverse is $z = \log x$ with the "Jacobian" $\partial z / \partial x = 1/x$, so the formula predicts

$$ \log p_x(x) = \log p_z(\log x) + \log \frac{1}{x} $$

Below we compare this with a histogram of transformed samples, and with what we would get if we forgot the Jacobian term.

In [ ]:
base = torch.distributions.Normal(0., 1.)

# Generative direction: sample z and transform it, x = f(z) = exp(z)
n_samples = 100000
z = base.sample((n_samples,))
x = torch.exp(z)

# Density of x from the change of variables formula
x_grid = torch.linspace(0.01, 20, 4000)
z_grid = torch.log(x_grid)                    # normalizing direction: z = f^{-1}(x) = log(x)
log_det = -torch.log(x_grid)                  # log|dz/dx| = log(1/x)
p_x = torch.exp(base.log_prob(z_grid) + log_det)
p_x_no_jacobian = torch.exp(base.log_prob(z_grid))

print(f'Integral of the density with Jacobian: {torch.trapezoid(p_x, x_grid).item():.3f}')
print(f'Integral of the density without Jacobian: {torch.trapezoid(p_x_no_jacobian, x_grid).item():.3f}')

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))

z_plot = torch.linspace(-4, 4, 200)
axs[0].hist(z.numpy(), bins=80, range=(-4, 4), density=True, alpha=0.5, label='Samples')
axs[0].plot(z_plot, base.log_prob(z_plot).exp(), color='black', label='$p_z(z)$')
axs[0].set(xlabel='z', title='Base distribution')
axs[0].legend()

axs[1].plot(z_plot, torch.exp(z_plot))
axs[1].set(xlabel='z', ylabel='x', ylim=(0, 6), title='Transformation $x = f(z) = e^z$')

# Histogram normalized by the total number of samples (also those outside of the plotted range)
bins = np.linspace(0, 6, 81)
axs[2].hist(x.numpy(), bins=bins, weights=np.full(n_samples, 1 / (n_samples * (bins[1] - bins[0]))), alpha=0.5, label='Samples')
axs[2].plot(x_grid, p_x, color='black', label='Change of variables')
axs[2].plot(x_grid, p_x_no_jacobian, color='red', ls='--', label='Without Jacobian')
axs[2].set(xlabel='x', xlim=(0, 6), title='Transformed distribution')
axs[2].legend()
plt.show()

The histogram of the transformed samples follows the density from the change of variables formula. Without the Jacobian term the curve has the wrong shape and is not even normalized: where the transformation stretches space ($x > 1$), the probability is spread over a larger interval and the density has to go down, and where it compresses space ($x < 1$), the density has to go up.

In this example we chose the transformation ourselves. In a normalizing flow, $f$ is a neural network with trainable parameters, and we train it such that $p_x$ matches the distribution of our data. For this, the transformation has to be invertible and its Jacobian determinant has to be cheap to compute. The following architectures are different ways to achieve this.

## 2. RealNVP (Real Non-Volume Preserving)

We will implement the **Affine Coupling Layer**, a key building block of RealNVP. It splits the input $x$ into two parts $(x_a, x_b)$ and transforms $x_b$ based on $x_a$.

$$ y_a = x_a $$
$$ y_b = x_b \odot \exp(s(x_a)) + t(x_a) $$

Here, $s$ (scale) and $t$ (translation) are neural networks. This transformation is easily invertible and its Jacobian determinant is simple to compute (it's triangular).

### Task: Density Estimation on 'Two Moons'

We will learn the distribution of the classic 'Two Moons' dataset.

In [ ]:
# Generate Data
def get_data(n_samples=2000):
    X, _ = make_moons(n_samples=n_samples, noise=0.05)
    return torch.tensor(X, dtype=torch.float32)

X_train = get_data()
X_test = get_data()  # independent samples to evaluate the trained models

plt.figure(figsize=(6, 6))
plt.scatter(X_train[:, 0], X_train[:, 1], s=5)
plt.title('Target Distribution: Two Moons')
plt.show()

### Implementing the Affine Coupling Layer

We need a neural network to predict the scale $s$ and translation $t$ parameters.

In [ ]:
class CouplingLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, mask):
        super().__init__()
        # Registering the mask as a buffer makes it part of the module (e.g. for .to(device))
        self.register_buffer('mask', mask)
        # Neural network to predict s and t
        # Input is masked input, output is concatenation of s and t
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(), # Tanh is often more stable for flows
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim * 2) # Output s and t
        )
    
    def forward(self, x):
        # x: [batch, input_dim]
        
        # Mask the input: x_a stays constant, x_b is transformed
        # If mask is [1, 0], x * mask gives [x1, 0]
        x_masked = x * self.mask
        
        # Predict s and t based on x_masked
        st = self.net(x_masked)
        s, t = st.chunk(2, dim=1)
        
        # We only apply transformation to the unmasked part.
        # To ensure stability, we often use tanh or clamp for s
        s = torch.tanh(s) 
        
        # Apply affine transformation:
        # y = x * exp(s) + t
        # But we only want to apply this where mask is 0.
        # Where mask is 1, y = x.
        
        # Scaling factor: exp(s) * (1 - mask) + mask
        # If mask=1: factor = 1. If mask=0: factor = exp(s)
        scale = torch.exp(s) * (1 - self.mask) + self.mask
        
        # Translation: t * (1 - mask)
        translation = t * (1 - self.mask)
        
        y = x * scale + translation
        
        # Log Determinant of Jacobian
        # Since it's diagonal (or triangular), it's sum of log diagonal entries.
        # log_det = sum(log(scale)) = sum(s * (1 - mask))
        log_det = torch.sum(s * (1 - self.mask), dim=1)
        
        return y, log_det
    
    def inverse(self, y):
        # Inverse transformation
        # x = (y - t) * exp(-s)
        # The masked part is unchanged by the layer, so we can recompute the same s and t from y.
        
        y_masked = y * self.mask
        st = self.net(y_masked)
        s, t = st.chunk(2, dim=1)
        s = torch.tanh(s)
        
        scale = torch.exp(s) * (1 - self.mask) + self.mask
        translation = t * (1 - self.mask)
        
        x = (y - translation) / scale
        return x

### Building the RealNVP Model

A single coupling layer only transforms half the variables. To transform all variables, we stack multiple layers and alternate the masks.

In [ ]:
class RealNVP(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        
        # Alternate masks: [1, 0] then [0, 1]
        mask = torch.tensor([1, 0], dtype=torch.float32)
        
        for i in range(n_layers):
            self.layers.append(CouplingLayer(input_dim, hidden_dim, mask))
            mask = 1 - mask # Flip mask
            
    def forward(self, x):
        log_det_sum = 0
        for layer in self.layers:
            x, log_det = layer(x)
            log_det_sum += log_det
        return x, log_det_sum
    
    def inverse(self, z):
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z

## 3. Training

We minimize the **Negative Log Likelihood (NLL)**, the same loss as for the Gaussian model at the end of Notebook 2:

$$ \mathcal{L} = - \mathbb{E}_{x \sim p_{data}} [\log p_x(x)] $$

Using the change of variables (with $z = f^{-1}(x)$, which is what `forward` computes):
$$ \log p_x(x) = \log p_z(z) + \log |\det J_{f^{-1}}(x)| $$

Since the flow is a composition of layers, the log-determinants of the individual layers $J_i$ simply add up:
$$ \mathcal{L} = - \mathbb{E} \left[ \log p_z(f^{-1}(x)) + \sum_i \log |\det J_i| \right] $$

For reference: the best possible value of this loss is the entropy of the true distribution. For the two moons with noise $\sigma = 0.05$ this is $\approx 0.29$.

In [ ]:
input_dim = 2
hidden_dim = 64
flow = RealNVP(input_dim, hidden_dim, n_layers=6)
optimizer = optim.Adam(flow.parameters(), lr=0.001)

# Standard Normal Prior
prior = torch.distributions.MultivariateNormal(torch.zeros(2), torch.eye(2))

losses = []
for epoch in range(500):
    optimizer.zero_grad()
    
    # Forward pass: x -> z
    z, log_det = flow(X_train)
    
    # Calculate Loss: - (log_prob_prior(z) + log_det)
    log_prob_z = prior.log_prob(z)
    loss = -torch.mean(log_prob_z + log_det)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

plt.plot(losses)
plt.title('NLL Loss')
plt.show()

## 4. Visualization

1.  **Latent Space**: Transform data $x$ to $z$. Should look Gaussian.
2.  **Sampling**: Sample $z \sim \mathcal{N}(0, I)$, transform to $x$. Should look like moons.

In [ ]:
# 1. Latent Space
with torch.no_grad():
    z_transformed, _ = flow(X_train)
    z_transformed = z_transformed.numpy()

# 2. Sampling
with torch.no_grad():
    z_sample = prior.sample((2000,))
    x_generated = flow.inverse(z_sample).numpy()

fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].scatter(X_train[:, 0], X_train[:, 1], s=5)
axs[0].set_title('Original Data (X)')

axs[1].scatter(z_transformed[:, 0], z_transformed[:, 1], s=5, color='orange')
axs[1].set_title('Transformed to Latent Space (Z)')
axs[1].set_xlim(-4, 4)
axs[1].set_ylim(-4, 4)

axs[2].scatter(x_generated[:, 0], x_generated[:, 1], s=5, color='green')
axs[2].set_title('Generated Samples from Flow')

plt.show()

### Layer by Layer

How does the flow turn two moons into a Gaussian? We can apply the coupling layers one at a time and look at the intermediate results. Every point keeps its color, so you can follow where it goes. Note that every layer only moves the points along one axis (the other coordinate is copied), alternating between the two axes.

In [ ]:
colors = X_train[:, 0].numpy()     # color of every point: its original x_1 coordinate
n_layers = len(flow.layers)

fig, axs = plt.subplots(1, n_layers + 1, figsize=(3 * (n_layers + 1), 3.3))
x = X_train
with torch.no_grad():
    for i in range(n_layers + 1):
        axs[i].scatter(x[:, 0], x[:, 1], s=2, c=colors, cmap='viridis')
        axs[i].set_title('Data (X)' if i == 0 else f'After layer {i}')
        axs[i].set_xlim(-5, 5)
        axs[i].set_ylim(-5, 5)
        if i < n_layers:
            x, _ = flow.layers[i](x)     # apply the next coupling layer
plt.tight_layout()
plt.show()

### The Learned Density

A flow does not only generate samples. It also gives us the exact, normalized density $p_x(x)$ at every point. This is what distinguishes flows from many other generative models, and it is the property we need for inference later. Here we evaluate the density on a grid.

In [ ]:
def flow_log_prob(model, x):
    # log p_x(x) = log p_z(z) + log|det dz/dx|
    z, log_det = model(x)
    return prior.log_prob(z) + log_det

def plot_density(model, ax, n_grid=200, extent=(-1.5, 2.5, -1.0, 1.5)):
    x1 = torch.linspace(extent[0], extent[1], n_grid)
    x2 = torch.linspace(extent[2], extent[3], n_grid)
    grid = torch.cartesian_prod(x1, x2)             # all points (x1, x2) of the grid
    with torch.no_grad():
        density = flow_log_prob(model, grid).exp().reshape(n_grid, n_grid)
    ax.imshow(density.T, origin='lower', extent=extent, aspect='auto', cmap='magma')

fig, ax = plt.subplots(figsize=(7, 5))
plot_density(flow, ax)
ax.set_title('Density $p_x(x)$ of the trained flow')
plt.show()

## 5. Testing a Flow

A flow with a bug can still produce a decreasing loss and reasonable-looking plots. If, for example, the log-determinant is wrong, the "density" is not normalized and the loss can become arbitrarily small. It is therefore good practice to test every new flow. There are three simple tests, analogous to the gradient check in Notebook 1:

1. **Inverse:** `inverse(forward(x))` has to return `x`.
2. **Jacobian:** the log-determinant returned by `forward` has to agree with the determinant of the Jacobian matrix computed by autograd.
3. **Normalization:** the density has to integrate to one.

Our models use single precision (`float32`), so we cannot expect perfect agreement: errors up to about $10^{-3}$ are rounding errors, while a bug typically results in errors of order one. The integral is computed on a grid, so it is only accurate to about two decimal places.

In [ ]:
def test_flow(model, x):
    with torch.no_grad():
        z, log_det = model(x)
        x_reconstructed = model.inverse(z)

    # 1. Inverse
    inverse_error = (x_reconstructed - x).abs().max().item()

    # 2. Jacobian: compute the full matrix dz/dx with autograd for a few points
    log_det_error = 0.
    for i in range(5):
        jacobian = torch.autograd.functional.jacobian(lambda x_i: model(x_i[None])[0][0], x[i])
        log_det_autograd = torch.linalg.slogdet(jacobian)[1]     # log|det J|
        log_det_error = max(log_det_error, (log_det_autograd - log_det[i]).abs().item())

    # 3. Normalization: integrate the density on a fine grid over a large box
    n_grid = 600
    x1 = torch.linspace(-3, 4, n_grid)
    x2 = torch.linspace(-3, 3, n_grid)
    grid = torch.cartesian_prod(x1, x2)
    with torch.no_grad():
        density = flow_log_prob(model, grid).exp()
    integral = density.sum().item() * (x1[1] - x1[0]).item() * (x2[1] - x2[0]).item()

    print(f'Largest error of the inverse: {inverse_error:.1e}, '
          f'largest error of the log-determinant: {log_det_error:.1e}, '
          f'integral of the density: {integral:.3f}')

test_flow(flow, X_test)

## 6. Advanced Architectures: MAF and NSF

RealNVP is simple but has limited expressivity: each layer only applies an affine transformation, and only to half of the variables (you can see the consequences in the density plot above). We will now explore two more advanced architectures: **Masked Autoregressive Flow (MAF)** and **Neural Spline Flow (NSF)**.

### Masked Autoregressive Flow (MAF)

MAF uses the autoregressive property:
$$ p(x) = \prod_{i=1}^D p(x_i | x_{1:i-1}) $$

Each dimension $x_i$ is transformed based on previous dimensions. We can implement this efficiently using a **Masked Autoencoder (MADE)**, which masks the weights of a standard MLP to respect the autoregressive order.

The transformation is affine (like RealNVP), but the parameters depend on all previous variables:
$$ z_i = \left(x_i - \mu_i(x_{1:i-1})\right) \cdot \exp(-\alpha_i(x_{1:i-1})) $$

Because $\mu_i$ and $\alpha_i$ only depend on $x_{1:i-1}$, the Jacobian $\partial z / \partial x$ is triangular and $\log |\det J| = -\sum_i \alpha_i$.

- **Density evaluation** ($x \to z$) is fast: all $\mu_i, \alpha_i$ are computed in a single pass through the MADE.
- **Sampling** ($z \to x$) is slow: $x_i = z_i \exp(\alpha_i) + \mu_i$ requires $x_{1:i-1}$, so we need $D$ sequential passes.

(The *Inverse Autoregressive Flow*, IAF, makes the opposite choice: fast sampling, slow density evaluation.)

In [ ]:
# Masked Linear Layer
class MaskedLinear(nn.Linear):
    def __init__(self, in_features, out_features, bias=True, mask=None):
        super().__init__(in_features, out_features, bias)
        self.register_buffer('mask', mask)
    
    def forward(self, input):
        return F.linear(input, self.weight * self.mask, self.bias)

# MADE Network
class MADE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim_multiplier=2):
        super().__init__()
        self.input_dim = input_dim
        
        # Create masks
        # Every unit gets a "degree" m: the index of the last input it is allowed to depend on.
        # Input layer: 0, 1, ..., D-1
        m_input = np.arange(input_dim)
        
        # Hidden layers: degrees 0, ..., D-2 (a unit with degree m sees the inputs 0...m)
        m_hidden_1 = np.arange(hidden_dim) % max(1, input_dim - 1)
        m_hidden_2 = np.arange(hidden_dim) % max(1, input_dim - 1)
        
        # Masks construction
        # M_1: hidden_1 vs input
        mask1 = torch.from_numpy(m_hidden_1[:, None] >= m_input[None, :]).float()
        # M_2: hidden_2 vs hidden_1
        mask2 = torch.from_numpy(m_hidden_2[:, None] >= m_hidden_1[None, :]).float()
        # M_out: output vs hidden_2
        # The outputs for dimension i may only depend on the inputs 0...i-1,
        # i.e. only on hidden units with degree < i (strict inequality!)
        # We have multiple outputs per dimension (e.g. shift and scale), ordered as
        # [mu_0, ..., mu_{D-1}, alpha_0, ..., alpha_{D-1}], so that we can split them with chunk().
        m_output = np.tile(np.arange(input_dim), output_dim_multiplier)
        mask3 = torch.from_numpy(m_output[:, None] > m_hidden_2[None, :]).float()
        
        self.net = nn.Sequential(
            MaskedLinear(input_dim, hidden_dim, mask=mask1),
            nn.ReLU(),
            MaskedLinear(hidden_dim, hidden_dim, mask=mask2),
            nn.ReLU(),
            MaskedLinear(hidden_dim, input_dim * output_dim_multiplier, mask=mask3)
        )

    def forward(self, x):
        return self.net(x)

class MAF(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(MADE(input_dim, hidden_dim, output_dim_multiplier=2))

    def get_params(self, made, x):
        # MADE outputs the parameters for each dimension based on the previous dimensions
        mu, log_alpha = made(x).chunk(2, dim=1)
        # As for RealNVP, we bound the log-scale for stability
        log_alpha = torch.tanh(log_alpha)
        return mu, log_alpha
            
    def forward(self, x):
        # Normalizing direction x -> z (fast): z_i = (x_i - mu_i(x_{<i})) * exp(-alpha_i(x_{<i}))
        log_det_sum = 0
        for made in self.layers:
            mu, log_alpha = self.get_params(made, x)
            
            x = (x - mu) * torch.exp(-log_alpha)
            log_det_sum += -torch.sum(log_alpha, dim=1)
            
            # Flip for mixing dimensions between layers
            x = x.flip(dims=[1])
            
        return x, log_det_sum

    # Inverse (Sampling) is slow for MAF (sequential)
    def inverse(self, z):
        for made in reversed(self.layers):
            # Undo the flip
            z = z.flip(dims=[1])
            # We need to solve x_i one by one
            x = torch.zeros_like(z)
            for i in range(z.shape[1]):
                mu, log_alpha = self.get_params(made, x)
                # z_i = (x_i - mu_i) * exp(-alpha_i)
                # x_i = z_i * exp(alpha_i) + mu_i
                x[:, i] = z[:, i] * torch.exp(log_alpha[:, i]) + mu[:, i]
            # The output of this layer is the input of the next (previous) one
            z = x
        return z

### Neural Spline Flow (NSF)

Affine transformations ($y = ax+b$) are limited. **Spline Flows** replace them by a flexible, monotonic, piecewise function.

A **Rational Quadratic Spline (RQS)** divides the interval $[-B, B]$ into $K$ bins. The bin edges $(x_k, y_k)$ are called knots. In each bin, the function is the ratio of two quadratic polynomials. It is fully specified by the bin widths, the bin heights and the (positive) derivatives $d_k$ at the knots, which makes it monotonic (hence invertible) and differentiable. With $s_k = (y_{k+1} - y_k) / (x_{k+1} - x_k)$ and $\xi = (x - x_k) / (x_{k+1} - x_k)$:

$$ y = y_k + \frac{(y_{k+1} - y_k) \left[ s_k \xi^2 + d_k \xi (1 - \xi) \right]}{s_k + (d_{k+1} + d_k - 2 s_k)\, \xi (1 - \xi)} $$

Both the derivative (needed for the Jacobian) and the inverse (the solution of a quadratic equation) are available in closed form. Outside of $[-B, B]$ the transformation is the identity ("linear tails").

The implementation of this function is mostly bookkeeping (finding the bin of every input, gathering its parameters, evaluating the formula above), so we moved it to the file `flow_utils.py` next to this notebook. Have a look if you are interested in the details. Here we import it and first look at a single spline.

In [ ]:
from flow_utils import rational_quadratic_spline, spline_knots

# One spline with K = 8 bins and random parameters
# (in a flow, these parameters are the outputs of a neural network)
n_bins = 8
unnorm_w, unnorm_h, unnorm_d = torch.randn(n_bins), torch.randn(n_bins), torch.randn(n_bins - 1)

x = torch.linspace(-4, 4, 1000)
params = [p.expand(len(x), -1) for p in (unnorm_w, unnorm_h, unnorm_d)]   # same parameters for every input
y, log_det = rational_quadratic_spline(x, *params)
x_reconstructed, _ = rational_quadratic_spline(y, *params, inverse=True)
print(f'Largest error of the inverse: {(x_reconstructed - x).abs().max().item():.1e}')

_, _, knots_x, knots_y = spline_knots(unnorm_w, unnorm_h)

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
axs[0].plot(x, y, label='Spline')
axs[0].plot(knots_x, knots_y, 'o', color='red', label='Knots')
axs[0].plot(x, x, color='gray', ls=':', label='Identity')
axs[0].axvspan(-3, 3, color='gray', alpha=0.1, label='Spline interval $[-B, B]$')
axs[0].set(xlabel='Input x', ylabel='Output y', title='Rational quadratic spline')
axs[0].legend()

axs[1].plot(x, log_det.exp())
axs[1].plot(knots_x, np.interp(knots_x, x, log_det.exp()), 'o', color='red')
axs[1].axvspan(-3, 3, color='gray', alpha=0.1)
axs[1].set(xlabel='Input x', ylabel='dy/dx', yscale='log', title='Derivative (always positive, continuous at the knots)')
plt.show()

The spline is smooth, strictly increasing and passes through all knots. Its shape is controlled by $3K - 1$ numbers (here 23), compared to two numbers for an affine transformation.

Now we build the `SplineCouplingLayer`: a coupling layer like in RealNVP, but the neural network predicts the parameters of a spline (widths, heights and derivatives) for every transformed dimension.

In [ ]:
class SplineCouplingLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, mask, n_bins=8):
        super().__init__()
        self.register_buffer('mask', mask)
        self.n_bins = n_bins
        
        # Predict K bin widths, K bin heights and K-1 derivatives for each dimension
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim * (3 * n_bins - 1))
        )

    def transform(self, x, inverse):
        # The masked part is unchanged by the layer, so the spline parameters
        # are identical in the forward and in the inverse direction.
        x_masked = x * self.mask
        params = self.net(x_masked).view(x.shape[0], x.shape[1], 3 * self.n_bins - 1)
        
        # Split params
        unnorm_w = params[..., :self.n_bins]
        unnorm_h = params[..., self.n_bins:2 * self.n_bins]
        unnorm_d = params[..., 2 * self.n_bins:]
        
        # Apply spline to x where mask is 0
        # To make it vectorized, we calculate for all, then mix.
        # Note: We treat each dim independently given the params
        y_full, log_det_full = rational_quadratic_spline(x, unnorm_w, unnorm_h, unnorm_d, inverse=inverse)
        
        # Apply mask
        y = x * self.mask + y_full * (1 - self.mask)
        log_det = torch.sum(log_det_full * (1 - self.mask), dim=1)
        
        return y, log_det

    def forward(self, x):
        return self.transform(x, inverse=False)

    def inverse(self, y):
        x, _ = self.transform(y, inverse=True)
        return x

class NSF(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        mask = torch.tensor([1, 0], dtype=torch.float32)
        for _ in range(n_layers):
            self.layers.append(SplineCouplingLayer(input_dim, hidden_dim, mask))
            mask = 1 - mask
            
    def forward(self, x):
        log_det_sum = 0
        for layer in self.layers:
            x, log_det = layer(x)
            log_det_sum += log_det
        return x, log_det_sum
        
    def inverse(self, z):
        for layer in reversed(self.layers):
            z = layer.inverse(z)
        return z

## 7. Comparison Training

We will now train RealNVP, MAF, and NSF side-by-side. To compare them fairly, we also evaluate the NLL on independent test samples: a training loss below the entropy of the true distribution ($\approx 0.29$) is a sign of overfitting.

After training, every model has to pass the tests of Section 5.

In [ ]:
models = {
    'RealNVP': RealNVP(2, 64, n_layers=6),
    'MAF': MAF(2, 64, n_layers=6),
    'NSF': NSF(2, 64, n_layers=6)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")
    optimizer = optim.Adam(model.parameters(), lr=0.002)
    loss_hist = []
    
    for epoch in range(1000):
        optimizer.zero_grad()
        loss = -torch.mean(flow_log_prob(model, X_train))
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
    
    # NLL on independent test data
    with torch.no_grad():
        test_loss = -torch.mean(flow_log_prob(model, X_test)).item()
    
    results[name] = {'model': model, 'loss': loss_hist}
    print(f"Final Training Loss: {loss_hist[-1]:.4f}, Test Loss: {test_loss:.4f}")
    test_flow(model, X_test)

# Visualization
fig, axs = plt.subplots(3, 3, figsize=(15, 14))
z_sample = prior.sample((2000,))

for i, (name, res) in enumerate(results.items()):
    # Plot Loss
    axs[0, i].plot(res['loss'])
    axs[0, i].set_title(f'{name} Loss')
    axs[0, i].set_xlabel('Epoch')
    axs[0, i].set_ylim(0, 3)
    
    # Plot Samples
    # (MAF sampling needs one pass per dimension, which is no problem in 2D)
    with torch.no_grad():
        x_gen = res['model'].inverse(z_sample).numpy()
            
    axs[1, i].scatter(x_gen[:, 0], x_gen[:, 1], s=5, color='green')
    axs[1, i].set_title(f'{name} Samples')
    axs[1, i].set_xlim(-1.5, 2.5)
    axs[1, i].set_ylim(-1, 1.5)

    # Plot Density
    plot_density(res['model'], axs[2, i])
    axs[2, i].set_title(f'{name} Density')

plt.tight_layout()
plt.show()

## 8. Conditional Flows: the Bridge to Simulation-Based Inference

So far, each flow describes one fixed distribution $p(x)$. In Notebook 2 we made the parameters of a Gaussian depend on an input and obtained a *conditional* distribution $p(D|z)$. We can do the same with a flow: we give the networks inside the coupling layers an additional input, the **context** $c$. Every value of $c$ then results in a different transformation and therefore in a different distribution: the flow models $p(x|c)$.

Nothing else changes. The transformation is still invertible in $x$ and the Jacobian is the same as before, because the context is just one more input of the networks $s$ and $t$, like the unchanged half $x_a$.

**Example:** two moons which are rotated by an angle $\varphi$. The angle is the context. The training data are pairs $(\varphi_i, x_i)$, where every sample has its own angle, and the loss is the same negative log-likelihood as before, now of $p(x|\varphi)$.

In [ ]:
class ConditionalCouplingLayer(nn.Module):
    def __init__(self, input_dim, context_dim, hidden_dim, mask):
        super().__init__()
        self.register_buffer('mask', mask)
        # The only difference to the CouplingLayer: the network has additional inputs for the context
        self.net = nn.Sequential(
            nn.Linear(input_dim + context_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim * 2)
        )

    def scale_and_translation(self, x, context):
        # s and t depend on the masked input AND on the context
        st = self.net(torch.cat([x * self.mask, context], dim=1))
        s, t = st.chunk(2, dim=1)
        s = torch.tanh(s) * (1 - self.mask)
        t = t * (1 - self.mask)
        return s, t

    def forward(self, x, context):
        s, t = self.scale_and_translation(x, context)
        return x * torch.exp(s) + t, torch.sum(s, dim=1)

    def inverse(self, y, context):
        s, t = self.scale_and_translation(y, context)
        return (y - t) * torch.exp(-s)

class ConditionalRealNVP(nn.Module):
    def __init__(self, input_dim, context_dim, hidden_dim, n_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        mask = torch.tensor([1, 0], dtype=torch.float32)
        for i in range(n_layers):
            self.layers.append(ConditionalCouplingLayer(input_dim, context_dim, hidden_dim, mask))
            mask = 1 - mask # Flip mask

    def forward(self, x, context):
        log_det_sum = 0
        for layer in self.layers:
            x, log_det = layer(x, context)
            log_det_sum += log_det
        return x, log_det_sum

    def inverse(self, z, context):
        for layer in reversed(self.layers):
            z = layer.inverse(z, context)
        return z

In [ ]:
def simulate_rotated_moons(angle):
    # angle: [N, 1] in radians. Returns one sample of the rotated two moons for every angle.
    x = get_data(len(angle)) - torch.tensor([0.5, 0.25])   # shift the centre to the origin
    cos, sin = torch.cos(angle[:, 0]), torch.sin(angle[:, 0])
    return torch.stack([cos * x[:, 0] - sin * x[:, 1],
                        sin * x[:, 0] + cos * x[:, 1]], dim=1)

# Training data: pairs (angle_i, x_i), every sample has its own angle between 0 and 90 degrees
angle_train = torch.rand(4000, 1) * np.pi / 2
x_train_cond = simulate_rotated_moons(angle_train)

conditional_flow = ConditionalRealNVP(input_dim=2, context_dim=1, hidden_dim=64, n_layers=6)
optimizer = optim.Adam(conditional_flow.parameters(), lr=0.002)

for epoch in range(1000):
    optimizer.zero_grad()
    z, log_det = conditional_flow(x_train_cond, angle_train)
    loss = -torch.mean(prior.log_prob(z) + log_det)     # the same NLL loss as before
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

In [ ]:
angles_deg = [0, 30, 60, 90]
fig, axs = plt.subplots(2, 4, figsize=(16, 8))

for i, angle_deg in enumerate(angles_deg):
    angle = torch.full((2000, 1), np.deg2rad(angle_deg), dtype=torch.float32)

    # Samples from the simulator
    x_true = simulate_rotated_moons(angle)
    # Samples from the conditional flow: z ~ N(0, I), then z -> x with the angle as context
    with torch.no_grad():
        x_gen = conditional_flow.inverse(prior.sample((2000,)), angle).numpy()

    axs[0, i].scatter(x_true[:, 0], x_true[:, 1], s=5)
    axs[0, i].set_title(rf'Simulator, $\varphi = {angle_deg}^\circ$')
    axs[1, i].scatter(x_gen[:, 0], x_gen[:, 1], s=5, color='green')
    axs[1, i].set_title(rf'Conditional flow, $\varphi = {angle_deg}^\circ$')
    for ax in axs[:, i]:
        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)

plt.tight_layout()
plt.show()

The samples are not perfect (more layers or longer training would help), but a single network now represents a whole family of distributions, one for each value of the context.

**Why this matters for inference:** a simulator takes parameters $\theta$ and produces data $x$. Running it many times gives us pairs $(\theta_i, x_i)$, just like the pairs $(\varphi_i, x_i)$ above. With these pairs we can train a conditional flow in two ways:
- context $= \theta$, target $= x$: the flow learns the **likelihood** $p(x|\theta)$ of the simulator. This is *Neural Likelihood Estimation (NLE)*, and it is what we just did (the angle was the parameter, the point was the data).
- context $= x$, target $= \theta$: the flow learns the **posterior** $p(\theta|x)$ directly. This is *Neural Posterior Estimation (NPE)*.

Both are the topic of Notebook 4.

## Summary

- A normalizing flow is an invertible, trainable transformation between data $x$ and a latent variable $z$ with a simple distribution. The change of variables formula turns it into a normalized probability density.
- The art is to design transformations which are flexible, invertible, and have a cheap Jacobian determinant. Coupling layers (RealNVP, NSF) are fast in both directions. Autoregressive layers (MAF) are fast for density evaluation and slow for sampling.
- Flows are trained by minimizing the negative log-likelihood of the training data, exactly like the Gaussian model in Notebook 2.
- Always test a flow: inverse, Jacobian and normalization.
- With an additional context input, a flow models a conditional distribution $p(x|c)$.

**Next:** in Notebook 4 we use this conditional flow to build simulation-based inference from scratch. The notebooks after that solve the same problems with the packages `sbi`, `ltu-ili` and `sbikit`, which provide well-tested flows, training loops and samplers.